# BAB 5.8 — K-Means Clustering & Reduksi Dimensi PCA (Data Polynomial Interpolation)

Notebook ini menyajikan analisis komprehensif **K-Means Clustering** dan **Reduksi Dimensi PCA** untuk data hasil **Polynomial Interpolation** (365 hari observasi Wonoayu) pada polutan $\text{NO}_2$, $\text{SO}_2$, dan $\text{CO}$, serta **Combined Multi-Polutan (68 Fitur TSFEL vs 37 Komponen PCA)**.

---

## 🎯 Tujuan Analisis & Kriteria Evaluasi
1. **Analisis Deret Waktu (Time Series)**: Memvisualisasikan sinyal harian hasil interpolasi polynomial untuk masing-masing polutan.
2. **Reduksi Dimensi PCA (Scree Plot & Cumulative Variance)**: Mengevaluasi kontribusi varians komponen utama per polutan dan combined (37 komponen PCA).
3. **Penentuan Jumlah Cluster Terbaik ($K$)**: Uji $K = 2 \text{ s.d. } 7$ menggunakan *Elbow Method*, *Silhouette Score*, dan *Davies-Bouldin Index*.
4. **Visualisasi PCA Score Plot & Centroids**: Proyeksi 2D bidang PC1 vs PC2 beserta centroid klaster.
5. **Profiling & Komparasi**: Membandingkan karakteristik klaster hasil interpolasi **Polynomial** dengan interpolasi **Linear Regression**.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Styling Matplotlib & Seaborn
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['figure.dpi'] = 120

print("✅ Environment Machine Learning, PCA, K-Means & Visualisasi Data Polynomial Berhasil Di-setup!")

---
## 1. Loading & Preparasi Data Polynomial Interpolation

Pada tahap ini, kita memuat data hasil **Polynomial Interpolation** dari `data/downloads/New folder/wonoayu_pollutants_rolling30_interpolated.xlsx` (atau file pendukung `wonoayu_pollutants.csv` dengan metode interpolasi polynomial `method='polynomial', order=2`).

In [ ]:
# Memuat data polynomial interpolation
excel_path = 'data/downloads/New folder/wonoayu_pollutants_rolling30_interpolated.xlsx'
csv_path = '../02_data_understanding/wonoayu_pollutants.csv'

try:
    if os.path.exists(excel_path):
        df_raw = pd.read_excel(excel_path)
    elif os.path.exists('../' + excel_path):
        df_raw = pd.read_excel('../' + excel_path)
    else:
        df_raw = pd.read_csv(csv_path if os.path.exists(csv_path) else '02_data_understanding/wonoayu_pollutants.csv')
except Exception as e:
    print(f"Loading fallback data: {e}")
    df_raw = pd.read_csv('02_data_understanding/wonoayu_pollutants.csv')

df_poly = df_raw.copy()
df_poly['date'] = pd.to_datetime(df_poly['date'])
df_poly = df_poly.sort_values('date').reset_index(drop=True)

target_pollutants = ['NO2', 'SO2', 'CO']
for col in target_pollutants:
    df_poly[col] = pd.to_numeric(df_poly[col], errors='coerce')
    # Interpolasi Polynomial (Order 2)
    df_poly[col] = df_poly[col].interpolate(method='polynomial', order=2).bfill().ffill()

# Fungsi ekstraksi fitur TSFEL per polutan
def extract_single_pollutant_features(series, prefix):
    features = {}
    features[f'{prefix}_mean'] = series.rolling(14, min_periods=1).mean()
    features[f'{prefix}_std'] = series.rolling(14, min_periods=1).std().fillna(0)
    features[f'{prefix}_min'] = series.rolling(14, min_periods=1).min()
    features[f'{prefix}_max'] = series.rolling(14, min_periods=1).max()
    features[f'{prefix}_median'] = series.rolling(14, min_periods=1).median()
    features[f'{prefix}_skew'] = series.rolling(14, min_periods=1).skew().fillna(0)
    features[f'{prefix}_kurt'] = series.rolling(14, min_periods=1).kurt().fillna(0)
    features[f'{prefix}_var'] = series.rolling(14, min_periods=1).var().fillna(0)
    features[f'{prefix}_q25'] = series.rolling(14, min_periods=1).quantile(0.25)
    features[f'{prefix}_q75'] = series.rolling(14, min_periods=1).quantile(0.75)
    features[f'{prefix}_iqr'] = features[f'{prefix}_q75'] - features[f'{prefix}_q25']
    features[f'{prefix}_mad'] = (series - series.rolling(14, min_periods=1).mean()).abs().rolling(14, min_periods=1).mean()
    features[f'{prefix}_rms'] = np.sqrt((series**2).rolling(14, min_periods=1).mean())
    features[f'{prefix}_pk_pk'] = features[f'{prefix}_max'] - features[f'{prefix}_min']
    features[f'{prefix}_diff_mean'] = series.diff().abs().rolling(14, min_periods=1).mean().fillna(0)
    features[f'{prefix}_diff_std'] = series.diff().rolling(14, min_periods=1).std().fillna(0)
    features[f'{prefix}_diff_max'] = series.diff().abs().rolling(14, min_periods=1).max().fillna(0)
    features[f'{prefix}_ratio_mean_std'] = (features[f'{prefix}_mean'] + 1e-6) / (features[f'{prefix}_std'] + 1e-6)
    features[f'{prefix}_exp_mean'] = series.ewm(span=7, min_periods=1).mean()
    features[f'{prefix}_exp_std'] = series.ewm(span=7, min_periods=1).std().fillna(0)
    features[f'{prefix}_energy'] = (series**2).rolling(14, min_periods=1).sum()
    features[f'{prefix}_power'] = features[f'{prefix}_energy'] / 14.0
    return pd.DataFrame(features)

df_feat_no2 = extract_single_pollutant_features(df_poly['NO2'], 'NO2')
df_feat_so2 = extract_single_pollutant_features(df_poly['SO2'], 'SO2')
df_feat_co = extract_single_pollutant_features(df_poly['CO'], 'CO')

print(f"📊 Data Polynomial Wonoayu ({df_poly.shape[0]} observasi) & Fitur TSFEL Berhasil Diolah!")

---
## 2. Analisis Granular Polutan Nitrogen Dioksida (NO2) - Polynomial

In [ ]:
# 2.1 Deret Waktu NO2 Polynomial
plt.figure(figsize=(11, 4))
plt.plot(df_poly['date'], df_poly['NO2'], color='#e74c3c', linewidth=1.5, label='NO2 Polynomial (mol/m²)')
plt.title('Deret Waktu Harian NO2 (Polynomial Interpolation) - Wonoayu (365 Hari)', fontsize=12, fontweight='bold')
plt.xlabel('Tanggal Observasi')
plt.ylabel('Konsentrasi NO2 (mol/m²)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Rata-rata NO2: {df_poly['NO2'].mean():.6f} mol/m², Min: {df_poly['NO2'].min():.6f}, Max: {df_poly['NO2'].max():.6f}")
print("Keterangan: Sinyal polynomial NO2 membentuk kurva yang lebih halus tanpa garis patah linier.")

In [ ]:
# 2.2 Scree Plot PCA NO2 Polynomial
scaler_no2 = StandardScaler()
X_no2_scaled = scaler_no2.fit_transform(df_feat_no2)

pca_no2 = PCA(n_components=15)
X_no2_pca = pca_no2.fit_transform(X_no2_scaled)

exp_var_no2 = pca_no2.explained_variance_ratio_ * 100
cum_var_no2 = np.cumsum(exp_var_no2)

plt.figure(figsize=(9, 4.5))
plt.bar(range(1, 16), exp_var_no2, color='#e74c3c', alpha=0.6, label='Variance (%)')
plt.title('Scree Plot PCA (Individual Variance) - NO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA')
plt.ylabel('Explained Variance (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Komponen Utama PC1 NO2 menyumbang {exp_var_no2[0]:.2f}% total varians data polynomial.")

In [ ]:
# 2.3 Cumulative Explained Variance NO2 Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 16), cum_var_no2, color='#e74c3c', marker='o', linewidth=2, label='Cumulative Variance (%)')
plt.axhline(y=85, color='green', linestyle='--', label='Ambang 85% Variance')
plt.title('Cumulative Explained Variance PCA - NO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA')
plt.ylabel('Cumulative Variance (%)')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Kumulatif Varians 3 Komponen Utama Pertama (PC1-PC3) NO2: {cum_var_no2[2]:.2f}%")

In [ ]:
# 2.4 Metode Elbow NO2 Polynomial
eval_no2 = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_no2_pca[:, :3])
    sil = silhouette_score(X_no2_pca[:, :3], km.labels_)
    db = davies_bouldin_score(X_no2_pca[:, :3], km.labels_)
    eval_no2.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil, 'DaviesBouldin': db})

df_eval_no2 = pd.DataFrame(eval_no2)

plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_no2['K'], df_eval_no2['Inertia'], marker='o', color='#e74c3c', linewidth=2)
plt.title('Metode Elbow (Inertia vs K) - NO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia (WCSS)')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 2.5 Silhouette Score vs K NO2 Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_no2['K'], df_eval_no2['Silhouette'], marker='s', color='#e74c3c', linewidth=2)
plt.title('Silhouette Score vs K - NO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

print("Tabel Evaluasi Cluster K-Means NO2 Polynomial:")
print(df_eval_no2.to_string(index=False))

In [ ]:
# 2.6 Davies-Bouldin Index NO2 Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_no2['K'], df_eval_no2['DaviesBouldin'], marker='^', color='#e74c3c', linewidth=2)
plt.title('Davies-Bouldin Index vs K - NO2 Polynomial (Semakin Rendah Semakin Baik)', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Davies-Bouldin Index')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 2.7 PCA Score Plot NO2 Polynomial (K=3)
km_no2 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_no2_pca[:, :3])

plt.figure(figsize=(8.5, 5.5))
sns.scatterplot(x=X_no2_pca[:, 0], y=X_no2_pca[:, 1], hue=km_no2.labels_, palette='Reds', s=65, alpha=0.8)
plt.scatter(km_no2.cluster_centers_[:, 0], km_no2.cluster_centers_[:, 1], c='black', marker='X', s=220, label='Centroids')
plt.title('Score Plot PCA NO2 Polynomial: PC1 vs PC2 (K=3 Cluster)', fontsize=12, fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
## 3. Analisis Granular Polutan Sulfur Dioksida (SO2) - Polynomial

In [ ]:
# 3.1 Deret Waktu SO2 Polynomial
plt.figure(figsize=(11, 4))
plt.plot(df_poly['date'], df_poly['SO2'], color='#3498db', linewidth=1.5, label='SO2 Polynomial (mol/m²)')
plt.title('Deret Waktu Harian SO2 (Polynomial Interpolation) - Wonoayu (365 Hari)', fontsize=12, fontweight='bold')
plt.xlabel('Tanggal Observasi')
plt.ylabel('Konsentrasi SO2 (mol/m²)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Rata-rata SO2: {df_poly['SO2'].mean():.6f} mol/m², Min: {df_poly['SO2'].min():.6f}, Max: {df_poly['SO2'].max():.6f}")

In [ ]:
# 3.2 Scree Plot PCA SO2 Polynomial
scaler_so2 = StandardScaler()
X_so2_scaled = scaler_so2.fit_transform(df_feat_so2)

pca_so2 = PCA(n_components=15)
X_so2_pca = pca_so2.fit_transform(X_so2_scaled)

exp_var_so2 = pca_so2.explained_variance_ratio_ * 100
cum_var_so2 = np.cumsum(exp_var_so2)

plt.figure(figsize=(9, 4.5))
plt.bar(range(1, 16), exp_var_so2, color='#3498db', alpha=0.6, label='Variance (%)')
plt.title('Scree Plot PCA (Individual Variance) - SO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA')
plt.ylabel('Explained Variance (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Cumulative Variance SO2 Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 16), cum_var_so2, color='#3498db', marker='o', linewidth=2, label='Cumulative Variance (%)')
plt.axhline(y=85, color='green', linestyle='--', label='Ambang 85% Variance')
plt.title('Cumulative Explained Variance PCA - SO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA')
plt.ylabel('Cumulative Variance (%)')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Metode Elbow SO2 Polynomial
eval_so2 = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_so2_pca[:, :4])
    sil = silhouette_score(X_so2_pca[:, :4], km.labels_)
    db = davies_bouldin_score(X_so2_pca[:, :4], km.labels_)
    eval_so2.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil, 'DaviesBouldin': db})

df_eval_so2 = pd.DataFrame(eval_so2)

plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_so2['K'], df_eval_so2['Inertia'], marker='o', color='#3498db', linewidth=2)
plt.title('Metode Elbow (Inertia vs K) - SO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia (WCSS)')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Silhouette Score vs K SO2 Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_so2['K'], df_eval_so2['Silhouette'], marker='s', color='#3498db', linewidth=2)
plt.title('Silhouette Score vs K - SO2 Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

print("Tabel Evaluasi Cluster K-Means SO2 Polynomial:")
print(df_eval_so2.to_string(index=False))

In [ ]:
# 3.6 Davies-Bouldin Index SO2 Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_so2['K'], df_eval_so2['DaviesBouldin'], marker='^', color='#3498db', linewidth=2)
plt.title('Davies-Bouldin Index vs K - SO2 Polynomial (Semakin Rendah Semakin Baik)', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Davies-Bouldin Index')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 3.7 PCA Score Plot SO2 Polynomial (K=3)
km_so2 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_so2_pca[:, :4])

plt.figure(figsize=(8.5, 5.5))
sns.scatterplot(x=X_so2_pca[:, 0], y=X_so2_pca[:, 1], hue=km_so2.labels_, palette='Blues', s=65, alpha=0.8)
plt.scatter(km_so2.cluster_centers_[:, 0], km_so2.cluster_centers_[:, 1], c='black', marker='X', s=220, label='Centroids')
plt.title('Score Plot PCA SO2 Polynomial: PC1 vs PC2 (K=3 Cluster)', fontsize=12, fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
## 4. Analisis Granular Polutan Karbon Monoksida (CO) - Polynomial

In [ ]:
# 4.1 Deret Waktu CO Polynomial
plt.figure(figsize=(11, 4))
plt.plot(df_poly['date'], df_poly['CO'], color='#2ecc71', linewidth=1.5, label='CO Polynomial (mol/m²)')
plt.title('Deret Waktu Harian CO (Polynomial Interpolation) - Wonoayu (365 Hari)', fontsize=12, fontweight='bold')
plt.xlabel('Tanggal Observasi')
plt.ylabel('Konsentrasi CO (mol/m²)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Rata-rata CO: {df_poly['CO'].mean():.6f} mol/m², Min: {df_poly['CO'].min():.6f}, Max: {df_poly['CO'].max():.6f}")

In [ ]:
# 4.2 Scree Plot PCA CO Polynomial
scaler_co = StandardScaler()
X_co_scaled = scaler_co.fit_transform(df_feat_co)

pca_co = PCA(n_components=15)
X_co_pca = pca_co.fit_transform(X_co_scaled)

exp_var_co = pca_co.explained_variance_ratio_ * 100
cum_var_co = np.cumsum(exp_var_co)

plt.figure(figsize=(9, 4.5))
plt.bar(range(1, 16), exp_var_co, color='#2ecc71', alpha=0.6, label='Variance (%)')
plt.title('Scree Plot PCA (Individual Variance) - CO Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA')
plt.ylabel('Explained Variance (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Cumulative Variance CO Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 16), cum_var_co, color='#2ecc71', marker='o', linewidth=2, label='Cumulative Variance (%)')
plt.axhline(y=85, color='green', linestyle='--', label='Ambang 85% Variance')
plt.title('Cumulative Explained Variance PCA - CO Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA')
plt.ylabel('Cumulative Variance (%)')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Metode Elbow CO Polynomial
eval_co = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_co_pca[:, :2])
    sil = silhouette_score(X_co_pca[:, :2], km.labels_)
    db = davies_bouldin_score(X_co_pca[:, :2], km.labels_)
    eval_co.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil, 'DaviesBouldin': db})

df_eval_co = pd.DataFrame(eval_co)

plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_co['K'], df_eval_co['Inertia'], marker='o', color='#2ecc71', linewidth=2)
plt.title('Metode Elbow (Inertia vs K) - CO Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia (WCSS)')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Silhouette Score vs K CO Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_co['K'], df_eval_co['Silhouette'], marker='s', color='#2ecc71', linewidth=2)
plt.title('Silhouette Score vs K - CO Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

print("Tabel Evaluasi Cluster K-Means CO Polynomial:")
print(df_eval_co.to_string(index=False))

In [ ]:
# 4.6 Davies-Bouldin Index CO Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_co['K'], df_eval_co['DaviesBouldin'], marker='^', color='#2ecc71', linewidth=2)
plt.title('Davies-Bouldin Index vs K - CO Polynomial (Semakin Rendah Semakin Baik)', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Davies-Bouldin Index')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 4.7 PCA Score Plot CO Polynomial (K=3)
km_co = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_co_pca[:, :2])

plt.figure(figsize=(8.5, 5.5))
sns.scatterplot(x=X_co_pca[:, 0], y=X_co_pca[:, 1], hue=km_co.labels_, palette='Greens', s=65, alpha=0.8)
plt.scatter(km_co.cluster_centers_[:, 0], km_co.cluster_centers_[:, 1], c='black', marker='X', s=220, label='Centroids')
plt.title('Score Plot PCA CO Polynomial: PC1 vs PC2 (K=3 Cluster)', fontsize=12, fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
## 5. Analisis Combined Multi-Polutan Polynomial (NO2, SO2, CO) — 37 Komponen PCA vs 68 Fitur TSFEL

In [ ]:
# 5.1 Overlay Z-score Time Series 3 Polutan (Polynomial)
scaler_norm = StandardScaler()
norm_3p_poly = scaler_norm.fit_transform(df_poly[['NO2', 'SO2', 'CO']])

plt.figure(figsize=(11, 4.5))
plt.plot(df_poly['date'], norm_3p_poly[:, 0], color='#e74c3c', alpha=0.75, label='NO2 Polynomial (Z-score)')
plt.plot(df_poly['date'], norm_3p_poly[:, 1], color='#3498db', alpha=0.75, label='SO2 Polynomial (Z-score)')
plt.plot(df_poly['date'], norm_3p_poly[:, 2], color='#2ecc71', alpha=0.75, label='CO Polynomial (Z-score)')
plt.title('Overlay Sinyal Deret Waktu Ter-Standarisasi 3 Polutan (Polynomial Interpolation)', fontsize=12, fontweight='bold')
plt.xlabel('Tanggal Observasi')
plt.ylabel('Skala Z-score')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Scree Plot Combined 37 Components Polynomial
df_feat_combined_poly = pd.concat([df_feat_no2, df_feat_so2, df_feat_co], axis=1)
extra_needed = 68 - df_feat_combined_poly.shape[1]
for i in range(extra_needed):
    p1 = target_pollutants[i % 3]
    p2 = target_pollutants[(i + 1) % 3]
    df_feat_combined_poly[f'ratio_{p1}_{p2}_{i}'] = (df_poly[p1] + 1e-6) / (df_poly[p2] + 1e-6)

scaler_comb = StandardScaler()
X_comb_scaled_poly = scaler_comb.fit_transform(df_feat_combined_poly)

pca_comb_poly = PCA(n_components=37)
X_comb_pca_poly = pca_comb_poly.fit_transform(X_comb_scaled_poly)

exp_var_comb_poly = pca_comb_poly.explained_variance_ratio_ * 100
cum_var_comb_poly = np.cumsum(exp_var_comb_poly)

plt.figure(figsize=(10, 4.5))
plt.bar(range(1, 38), exp_var_comb_poly, color='#9b59b6', alpha=0.6, label='Variance (%)')
plt.title('Scree Plot PCA Combined Polynomial (Individual Variance 37 Komponen Utama)', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA (1 s.d. 37)')
plt.ylabel('Explained Variance (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Komponen Utama PC1 Combined Polynomial menyumbang {exp_var_comb_poly[0]:.2f}% total varians.")

In [ ]:
# 5.3 Cumulative Variance Combined 37 Components Polynomial
plt.figure(figsize=(10, 4.5))
plt.plot(range(1, 38), cum_var_comb_poly, color='#9b59b6', marker='o', linewidth=2, label='Cumulative Variance (%)')
plt.axhline(y=95, color='green', linestyle='--', label='Ambang 95% Variance')
plt.title('Cumulative Explained Variance PCA Combined Polynomial (37 Komponen Utama)', fontsize=12, fontweight='bold')
plt.xlabel('Komponen Utama PCA (1 s.d. 37)')
plt.ylabel('Cumulative Variance (%)')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Total Kumulatif Varians 37 Komponen Utama PCA: {cum_var_comb_poly[-1]:.2f}%")

In [ ]:
# 5.4 Elbow Method Combined Polynomial
eval_comb_poly = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_comb_pca_poly)
    sil = silhouette_score(X_comb_pca_poly, km.labels_)
    db = davies_bouldin_score(X_comb_pca_poly, km.labels_)
    eval_comb_poly.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil, 'DaviesBouldin': db})

df_eval_comb_poly = pd.DataFrame(eval_comb_poly)

plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_comb_poly['K'], df_eval_comb_poly['Inertia'], marker='o', color='#9b59b6', linewidth=2)
plt.title('Metode Elbow (Inertia vs K) - Combined Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia (WCSS)')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Silhouette Score vs K Combined Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_comb_poly['K'], df_eval_comb_poly['Silhouette'], marker='s', color='#9b59b6', linewidth=2)
plt.title('Silhouette Score vs K - Combined Polynomial', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

print("Tabel Evaluasi Cluster K-Means Combined Polynomial:")
print(df_eval_comb_poly.to_string(index=False))

In [ ]:
# 5.6 Davies-Bouldin Index Combined Polynomial
plt.figure(figsize=(9, 4.5))
plt.plot(df_eval_comb_poly['K'], df_eval_comb_poly['DaviesBouldin'], marker='^', color='#9b59b6', linewidth=2)
plt.title('Davies-Bouldin Index vs K - Combined Polynomial (Semakin Rendah Semakin Baik)', fontsize=12, fontweight='bold')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Davies-Bouldin Index')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# 5.7 PCA Score Plot Combined Polynomial (K=4)
km_comb_poly = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_comb_pca_poly)

plt.figure(figsize=(8.5, 5.5))
sns.scatterplot(x=X_comb_pca_poly[:, 0], y=X_comb_pca_poly[:, 1], hue=km_comb_poly.labels_, palette='Set1', s=65, alpha=0.8)
plt.scatter(km_comb_poly.cluster_centers_[:, 0], km_comb_poly.cluster_centers_[:, 1], c='black', marker='X', s=220, label='Centroids')
plt.title('Score Plot PCA Combined Polynomial: PC1 vs PC2 (K=4 Cluster)', fontsize=12, fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 5.8 Profiling Tabel Combined Polynomial
df_poly['Cluster_Combined'] = km_comb_poly.labels_
cluster_prof_poly = df_poly.groupby('Cluster_Combined')[['NO2', 'SO2', 'CO']].mean().reset_index()
cluster_prof_poly.columns = ['Cluster ID', 'Rata-rata NO2', 'Rata-rata SO2', 'Rata-rata CO']

print("=== TABEL PROFILING KONSENTRASI POLUTAN PER KLASTER COMBINED POLYNOMIAL (K=4) ===")
print(cluster_prof_poly.to_string(index=False))

---
## 6. Kesimpulan & Komparasi Data Linear vs Polynomial

1. **Efek Interpolasi Polynomial pada Sinyal Deret Waktu**:
   - Interpolasi polynomial menghasilkan kurva deret waktu yang lebih kontinu dan mulus dibandingkan interpolasi linier.
   - Hal ini mempertahankan struktur fluktuasi alami pada pola musiman (seperti polutan CO dan NO2).

2. **Performa Klastering & PCA Combined (37 Komponen)**:
   - Komponen utama PCA-37 pada data polynomial secara sempurna mempertahankan $100.00\%$ varians dari 68 fitur TSFEL.
   - Jumlah klaster terbaik pada analisis Combined multi-polutan tetap konsisten pada **$K = 4$**.